# Analyze Ra-Ap burst relationship

This notebook analyzes the relationship of RA-AP to the bursts.

## Initialization - don't change

In [ ]:
import sys
sys.path.append('..')

from param_classes.workflows.input_output import InputDataParams
from workflows.run_util.io import collect_paths_from_table

data_params = InputDataParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [ ]:
palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'AIS-AP': "#3150B7",
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Input-output settings

Base directory paths
 - `MAT_DIR` - base dir of the raw **mat** files, grouped into subfolders
 - `PROCESSED_DIR` - base dir of the preprocessed data and results

In [ ]:
MAT_DIR = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
PROCESSED_DIR = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed'

In [ ]:
input_param = {
    'mode': 'table',
    'table_path': f'{PROCESSED_DIR}\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

In [ ]:
data_params.raw_data.path= MAT_DIR
data_params.raw_data.extension='mat'
data_params.spike_features.path = f'{PROCESSED_DIR}\\SpikeFeatures'
data_params.spike_features.extension = 'csv'

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
output_path = f'{PROCESSED_DIR}\\RaAp_Burst_Analysis'
result_path = f'{output_path}\\01_results'
figure_path = f'{output_path}\\02_figures'
statistics_path = f'{output_path}\\03_statistics'

## Analysis parameters

Name of the spike types

In [ ]:
spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']

List of the groups in the order of display.

In [ ]:
group_list = [
    'FSI_L2-3',
    'RSI_L2-3',
    'PYR_L2-3',
    'FSI_L4-6',
    'RSI_L4-6',
    'PYR_L4-6'
]

List of groups to test for enrichment value.

In [ ]:
test_groups = [
    'FSI_L2-3',
    'RSI_L2-3',
    'FSI_L4-6',
    'RSI_L4-6',
]

Parameters for the burst detection
 - `max_burst_isi`: size of the ISI where the burst ends (interval between spikes of a burst must be less than this)
 - `min_burst_count`: minimum number of spikes considered as burst. Set `2` to consider duplets as burst

In [ ]:
max_burst_isi = 0.006
min_burst_count = 3

Parameters for local density analysis
 - `local_density_window`: window size for local density calculation

In [ ]:
local_density_window = 0.006

## Plot parameters

## Run analysis

Collect input files according to the table.

In [ ]:
input_files = collect_paths_from_table(data_params, input_param)
n_cells = len(input_files)

In [ ]:
from pathlib import Path
import pandas as pd
from workflows.burst_analysis import analyze_burst_probabilities
from workflows.burst_analysis import analyze_spike_neighborhood

results = []
for input_file in input_files:
    cell_data = input_file.data.load()

    cell_result = {
        'name': input_file.name,
        'group': input_file.cell_type
    }

    cell_result = cell_result | analyze_burst_probabilities(
        cell_data.spikes,
        max_isi=max_burst_isi,
        min_count=min_burst_count
    )
    cell_result = cell_result | analyze_spike_neighborhood(
        cell_data.spikes,
        window_size=local_density_window,
    )
    results.append(cell_result)

results_df = pd.DataFrame(results)

In [ ]:
from workflows.burst_analysis import plot_prob_change_per_cell
plot_prob_change_per_cell(
    data=results_df,
    group_names=group_list,
    palette=palette['cell_type'],
    save_path=f'{output_path}\\02_figures'
)

In [ ]:
from workflows.burst_analysis import plot_prob_change_per_group
plot_prob_change_per_group(
    data=results_df,
    group_names=group_list,
    palette=palette['cell_type'],
    save_path=f'{output_path}\\02_figures'
)

In [ ]:
from workflows.burst_analysis import plot_enrichment_ratio_per_group
plot_enrichment_ratio_per_group(
    data=results_df,
    group_names=group_list,
    palette=palette['cell_type'],
    feature_name='log_enrichment',
    save_path=f'{output_path}\\02_figures'
)

In [ ]:
from workflows.burst_analysis import test_enrichment_ratio
test_enrichment_ratio(
    data=results_df,
    test_groups=group_list,
    save_path=statistics_path
)

In [ ]:
Path(result_path).mkdir(exist_ok=True, parents=True)
results_df.to_excel(
    f'{result_path}/raap_burst_analysis.xlsx',
    index=False
)